# Data Extraction

**Purpose:** Extract raw text from all PDF files in `data/raw-data/` and save the output to `data/extracted/`.

**Pipeline Position:** `raw-data PDFs` → **[DATA EXTRACTION]** → `data/extracted/*.json`

---

### What this notebook does:
1. Reads all PDF files from `data/raw-data/`
2. Extracts text page-by-page using `PyMuPDF (fitz)`
3. Tags each page with metadata (bank name, filename, page number)
4. Saves extracted output as JSON files to `data/extracted/`

### Output saved to:
`data/extracted/<bank-filename>.json` — one JSON per PDF

---

In [1]:
import fitz
import os
import json
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
# Paths
RAW_DATA_DIR = Path("../../data/raw-data")
EXTRACTED_DIR = Path("../../data/extracted")

# Create output folder (only if it doesn't exist)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

# List all PDFs
pdf_files = sorted(RAW_DATA_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files:\n")
for f in pdf_files:
    print(f"  - {f.name}")

Found 16 PDF files:

  - ABL-Business-User-Guidelines.pdf
  - ABL-FAQs.pdf
  - Bank-Alfalah-FAQs-Account-Opening.pdf
  - Bank-Alfalah-FAQs-Personal-Loan.pdf
  - Bank-Alfalah-Self-Service-Banking.pdf
  - HBL-FAQs-Home-Remittance.pdf
  - HBL-FAQs.pdf
  - HBL-Islamic-Current-Account.pdf
  - HBL-Work-Conventional-Accounts.pdf
  - Meezan-Bank-FAQs-Debit-Cards.pdf
  - Meezan-Bank-FAQs-Digital-Account.pdf
  - Meezan-Bank-FAQs-Roshan-Apna-Ghar.pdf
  - Meezan-Bank-FAQs-Roshan-Digital-Account.pdf
  - Meezan-Bank-FAQs-Salaried.pdf
  - State-Bank-FAQs-History.pdf
  - State-Bank-FAQs.pdf


In [4]:
# Map filename prefix → Bank name
BANK_NAME_MAP = {
    "HBL": "Habib Bank Limited (HBL)",
    "Meezan": "Meezan Bank",
    "Bank-Alfalah": "Bank Alfalah",
    "ABL": "Allied Bank Limited (ABL)",
    "State-Bank": "State Bank of Pakistan (SBP)",
}

# Function to get bank name from filename
def get_bank_name(filename: str) -> str:
    """Return bank name based on filename prefix."""
    for prefix, name in BANK_NAME_MAP.items():
        if filename.startswith(prefix):
            return name
    return "Unknown Bank"

print("Bank name mapper ready")

Bank name mapper ready


In [6]:
# Function to extract text from PDF
def extract_text_from_pdf(pdf_path: Path) -> list[dict]:
    """
    Extract text from each page of a PDF.

    Returns:
        List of dicts — one dict per page with keys:
        - bank_name: str
        - source_file: str
        - page_number: int
        - total_pages: int
        - raw_text: str
    """
    pages = []
    bank_name = get_bank_name(pdf_path.stem)

    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")  # Plain text extraction
        pages.append({
            "bank_name": bank_name,
            "source_file": pdf_path.name,
            "page_number": page_num,
            "total_pages": total_pages,
            "raw_text": text
        })

    doc.close()
    return pages


print("Extraction function defined")

Extraction function defined


In [7]:
# Run extraction on all PDFs
all_extraction_results = {}

for pdf_path in pdf_files:
    print(f"Extracting: {pdf_path.name} ...", end=" ")
    pages = extract_text_from_pdf(pdf_path)
    all_extraction_results[pdf_path.stem] = pages
    print(f"{len(pages)} pages extracted")

print(f"\nExtraction complete — {len(all_extraction_results)} files processed")

Extracting: ABL-Business-User-Guidelines.pdf ... 7 pages extracted
Extracting: ABL-FAQs.pdf ... 19 pages extracted
Extracting: Bank-Alfalah-FAQs-Account-Opening.pdf ... 8 pages extracted
Extracting: Bank-Alfalah-FAQs-Personal-Loan.pdf ... 3 pages extracted
Extracting: Bank-Alfalah-Self-Service-Banking.pdf ... 5 pages extracted
Extracting: HBL-FAQs-Home-Remittance.pdf ... 2 pages extracted
Extracting: HBL-FAQs.pdf ... 5 pages extracted
Extracting: HBL-Islamic-Current-Account.pdf ... 4 pages extracted
Extracting: HBL-Work-Conventional-Accounts.pdf ... 6 pages extracted
Extracting: Meezan-Bank-FAQs-Debit-Cards.pdf ... 2 pages extracted
Extracting: Meezan-Bank-FAQs-Digital-Account.pdf ... 3 pages extracted
Extracting: Meezan-Bank-FAQs-Roshan-Apna-Ghar.pdf ... 2 pages extracted
Extracting: Meezan-Bank-FAQs-Roshan-Digital-Account.pdf ... 9 pages extracted
Extracting: Meezan-Bank-FAQs-Salaried.pdf ... 5 pages extracted
Extracting: State-Bank-FAQs-History.pdf ... 9 pages extracted
Extracting: 

In [11]:
# Preview first page of the first PDF
first_key = list(all_extraction_results.keys())[0]
first_page = all_extraction_results[first_key][0]

print(f"File   : {first_page['source_file']}")
print(f"Bank   : {first_page['bank_name']}")
print(f"Page   : {first_page['page_number']} / {first_page['total_pages']}")
print(f"\n--- Raw Text Preview (first 500 chars) ---")
print(first_page['raw_text'][:500])

File   : ABL-Business-User-Guidelines.pdf
Bank   : Allied Bank Limited (ABL)
Page   : 1 / 7

--- Raw Text Preview (first 500 chars) ---
 
Business Internet Banking 
User Guidelines 
Allied Bank Limited



In [12]:
# Save each file's pages as a separate JSON
for filename_stem, pages in all_extraction_results.items():
    output_path = EXTRACTED_DIR / f"{filename_stem}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved: {output_path}")

print(f"\nAll extracted data saved to: {EXTRACTED_DIR}")

💾 Saved: ..\data\extracted\ABL-Business-User-Guidelines.json
💾 Saved: ..\data\extracted\ABL-FAQs.json
💾 Saved: ..\data\extracted\Bank-Alfalah-FAQs-Account-Opening.json
💾 Saved: ..\data\extracted\Bank-Alfalah-FAQs-Personal-Loan.json
💾 Saved: ..\data\extracted\Bank-Alfalah-Self-Service-Banking.json
💾 Saved: ..\data\extracted\HBL-FAQs-Home-Remittance.json
💾 Saved: ..\data\extracted\HBL-FAQs.json
💾 Saved: ..\data\extracted\HBL-Islamic-Current-Account.json
💾 Saved: ..\data\extracted\HBL-Work-Conventional-Accounts.json
💾 Saved: ..\data\extracted\Meezan-Bank-FAQs-Debit-Cards.json
💾 Saved: ..\data\extracted\Meezan-Bank-FAQs-Digital-Account.json
💾 Saved: ..\data\extracted\Meezan-Bank-FAQs-Roshan-Apna-Ghar.json
💾 Saved: ..\data\extracted\Meezan-Bank-FAQs-Roshan-Digital-Account.json
💾 Saved: ..\data\extracted\Meezan-Bank-FAQs-Salaried.json
💾 Saved: ..\data\extracted\State-Bank-FAQs-History.json
💾 Saved: ..\data\extracted\State-Bank-FAQs.json

All extracted data saved to: ..\data\extracted


## 📊 Step 7 — Extraction Summary

In [ ]:
print("=" * 55)
print(f"{'FILE':<45} {'PAGES':>5}")
print("=" * 55)

total_pages = 0
for filename_stem, pages in all_extraction_results.items():
    n = len(pages)
    total_pages += n
    print(f"{filename_stem:<45} {n:>5}")

print("-" * 55)
print(f"{'TOTAL PAGES':<45} {total_pages:>5}")
print("=" * 55)
print(f"\n➡️  Next Step: Run 02_data_cleaning.ipynb")